<a href="https://colab.research.google.com/github/shikhar286/Agentic-AI-Portfolio-Shikhar-2026/blob/main/Autogen_FinincialReportAnalyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q ag2

import os
from google.colab import files
from autogen import UserProxyAgent, AssistantAgent, GroupChat, GroupChatManager

os.environ["OPENAI_API_KEY"] = input("Enter Your OPENAI_API_KEY")

llm_config = {"config_list" : [{'model' : 'gpt-4o-mini', 'api_key' : os.environ["OPENAI_API_KEY"]}], "temperature" : 0.2}

def make_agent(name,role):
  return AssistantAgent (name=name , llm_config = llm_config , system_message = role)

extractor_role = """You are an expert Financial Data Analyst. Your sole job is to read raw banking quarterly reports and cleanly extract the financial metrics.
Separate them into:
1. Income Metrics (NII, Net Profit, etc.)
2. Operational Metrics (ROA, ROE, Cost-to-Income)
Provide the extracted values along with their Quarter-on-Quarter (QoQ) changes exactly as listed in the report. Do not add narrative or interpretation."""

evaluator_role = """You are a Senior Risk Officer and Banking Compliance Auditor. Your job is to analyze the extracted metrics for systemic risks and regulatory breaches.
1. Assess Asset Quality Risk: Highlight rising Gross/Net NPAs and declining Provision Coverage.
2. Assess Operational/Sector Risk: Evaluate negative growth segments (e.g., Corporate/Treasury) and rising Cost-to-Income.
3. Verify Regulatory Compliance: Explicitly check if CRAR and LCR meet the minimum baseline requirements.
State clearly which metrics are 'Safe' and which are 'Critical Warnings'."""

summarizer_role = """You are a Financial Journalist and Investment Communication Expert. Your job is to compile the technical extractions and risk reports into a clean, concise Executive Summary for internal stakeholders.
Structure your final summary with:
- Executive Overview (High-level performance summary)
- Key Financial Highlights (Strengths)
- Key Operational & Compliance Risks (Weaknesses/Threats)
- Final Stakeholder Verdict (Is the bank overall stable, cautious, or failing?)
Ensure the tone is professional, clear, and perfectly action-oriented."""

# Create the 3 AssistantAgents using your factory function
financial_extractor = make_agent("Financial_Extractor", extractor_role)
risk_compliance_evaluator = make_agent("Risk_Compliance_Evaluator", evaluator_role)
executive_summarizer = make_agent("Executive_Summarizer", summarizer_role)

user_proxy = UserProxyAgent(name='CFO',human_input_mode='NEVER',code_execution_config=False)

REPORT = input("Share Bank Report:")

group_chat = GroupChat(agents=[financial_extractor, risk_compliance_evaluator, executive_summarizer, user_proxy],
messages=[],
max_round=6,
speaker_selection_method = 'auto')

manager = GroupChatManager(groupchat=group_chat , llm_config = llm_config)

result = user_proxy.initiate_chat(manager,message=f"""
Please analyze the following banking quarterly report and compile a complete compliance and risk summary.

### WORKFLOW INSTRUCTIONS FOR CHAT MANAGER:
1. First, **Financial_Extractor** must extract the baseline numeric values and changes from the raw text.
2. Next, **Risk_Compliance_Evaluator** must review those extracted numbers to flag asset health trends, operational dips, and baseline regulatory checks (CRAR and LCR limits).
3. Finally, **Executive_Summarizer** must take the risk evaluation and data breakdown to compile a structured investment briefing for leadership.

### RAW DATA REPORT TO PROCESS:
{REPORT}""")

files_content = '\n\n'.join(f"{m.get("name",m.get("role",''))} :\n {m.get("content","")}" for m in group_chat.messages)

filename = 'HDFC_Bank—Q3_FY2025_Quarterly_Report'
with open(filename,'w',encoding='utf-8') as f1:
 f1.write(files_content)


# Download
files.download(filename)